# WorldSim v3.1 — 9B GGUF Fix + 8-Model Extended Comparison
Fix 9B conversion, then compare 4 base + 4 fine-tuned across 108 prompts


## 1. Fix 9B FT GGUF


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, time, shutil, torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

GGUF_DIR = REPO_ROOT / "artifacts" / "gguf"
LLAMA_QUANTIZE = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-quantize"
CONVERT_SCRIPT = REPO_ROOT / "tools" / "llama.cpp" / "convert_hf_to_gguf.py"
LLAMA_SERVER = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-server"

outputs_9b_root = REPO_ROOT / "outputs" / "baseline" / "worldsim-v31-9b"
if outputs_9b_root.exists():
    run_dirs = sorted(outputs_9b_root.glob("run-*"))
    if run_dirs:
        OUTPUT_9B = run_dirs[-1]
        print(f"9B output dir: {OUTPUT_9B}")
    else:
        raise FileNotFoundError("No run directories found for 9B")
else:
    raise FileNotFoundError(f"9B output root not found: {outputs_9b_root}")

adapter_dir = OUTPUT_9B / "adapter"
merged_dir = OUTPUT_9B / "merged_bf16"
fp16_gguf = OUTPUT_9B / "worldsim-v31-qwen3.5-9b-f16.gguf"
q4_gguf = OUTPUT_9B / "worldsim-v31-qwen3.5-9b-q4_k_m.gguf"
final_gguf = GGUF_DIR / "worldsim-v31-qwen3.5-9b-q4_k_m.gguf"

assert adapter_dir.exists(), f"9B adapter not found: {adapter_dir}"
print(f"Adapter: {adapter_dir}")
print(f"Merged exists: {(merged_dir / 'config.json').exists()}")
print(f"FP16 GGUF exists: {fp16_gguf.exists()}")
print(f"Q4 GGUF exists: {q4_gguf.exists()}")

if merged_dir.exists() and not fp16_gguf.exists():
    print("Deleting broken merged dir from previous attempt...")
    shutil.rmtree(merged_dir)

if not (merged_dir / "config.json").exists():
    print("\n=== Merging 9B LoRA (CPU mode — no device offloading) ===", flush=True)
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    base = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3.5-9B-Base",
        torch_dtype=torch.bfloat16,
        device_map="cpu",
    )
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-9B-Base")

    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
    merged_dir.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    del merged, base
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    print("  Merged ✅", flush=True)
else:
    print("  Merged dir exists, skipping merge")

if not fp16_gguf.exists():
    print("\n=== HF → GGUF FP16 ===", flush=True)
    proc = subprocess.run(
        [sys.executable, str(CONVERT_SCRIPT), str(merged_dir), "--outfile", str(fp16_gguf), "--outtype", "f16"],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        print(f"STDERR: {proc.stderr[-500:]}")
        raise RuntimeError("GGUF conversion failed")
    print(f"  FP16: {fp16_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)

if not q4_gguf.exists():
    print("\n=== Quantize Q4_K_M ===", flush=True)
    proc = subprocess.run([str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), "Q4_K_M"], capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"STDERR: {proc.stderr[-500:]}")
        raise RuntimeError("Quantization failed")
    print(f"  Q4_K_M: {q4_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)

if not final_gguf.exists():
    shutil.copy2(q4_gguf, final_gguf)

print(f"\n✅ 9B FT GGUF: {final_gguf.name} ({final_gguf.stat().st_size / 1e6:.0f} MB)")


## 2. Model Inventory


In [ ]:
import random
import re
from collections import Counter, defaultdict

import yaml


def load_yaml(path: Path):
    with path.open(encoding='utf-8') as handle:
        return yaml.safe_load(handle)


situations = load_yaml(REPO_ROOT / 'config/situations.yaml')['situations']
personalities = load_yaml(REPO_ROOT / 'config/personalities.yaml')['personalities']
emotions = load_yaml(REPO_ROOT / 'config/emotions.yaml')['emotions']
social_situations = load_yaml(REPO_ROOT / 'config/social_situations.yaml')['social_situations']
group_situations = load_yaml(REPO_ROOT / 'config/group_situations.yaml')['group_situations']
trade_scenarios = load_yaml(REPO_ROOT / 'config/trade_scenarios.yaml')['trade_scenarios']
stress_sources = load_yaml(REPO_ROOT / 'config/stress_sources.yaml')['stress_sources']
deception_scenarios = load_yaml(REPO_ROOT / 'config/deception_scenarios.yaml')['deception_scenarios']
rumor_scenarios = load_yaml(REPO_ROOT / 'config/rumor_scenarios.yaml')['rumor_scenarios']
trauma_scenarios = load_yaml(REPO_ROOT / 'config/trauma_scenarios.yaml')['trauma_scenarios']
negotiation_scenarios = load_yaml(REPO_ROOT / 'config/negotiation_scenarios.yaml')['negotiation_scenarios']
culture_scenarios = load_yaml(REPO_ROOT / 'config/culture_scenarios.yaml')['culture_scenarios']
group_dissent_scenarios = load_yaml(REPO_ROOT / 'config/group_dissent_scenarios.yaml')['group_dissent_scenarios']

TEMPERAMENTS = [
    {'id': 'choleric', 'ns': 0.8, 'ha': 0.2, 'rd': 0.5, 'p': 0.7, 'keywords': 'bold, impulsive, dominant'},
    {'id': 'melancholic', 'ns': 0.2, 'ha': 0.8, 'rd': 0.6, 'p': 0.4, 'keywords': 'cautious, anxious, detail-oriented'},
    {'id': 'sanguine', 'ns': 0.6, 'ha': 0.3, 'rd': 0.8, 'p': 0.3, 'keywords': 'sociable, warm, optimistic'},
    {'id': 'phlegmatic', 'ns': 0.3, 'ha': 0.5, 'rd': 0.4, 'p': 0.8, 'keywords': 'calm, patient, persistent'},
]

BASE_HF_MODELS = {
    'base_08b': 'Qwen/Qwen3.5-0.8B-Base',
    'base_2b': 'Qwen/Qwen3.5-2B-Base',
    'base_4b': 'Qwen/Qwen3.5-4B-Base',
    'base_9b': 'Qwen/Qwen3.5-9B-Base',
}

MODELS = {}
model_configs = [
    ('base_08b', 'qwen3.5-0.8b-base-q4_k_m.gguf', '0.8B Base'),
    ('ft_08b', 'worldsim-v31-qwen3.5-0.8b-q4_k_m.gguf', '0.8B FT v3.1'),
    ('base_2b', 'qwen3.5-2b-base-q4_k_m.gguf', '2B Base'),
    ('ft_2b', 'worldsim-v31-qwen3.5-2b-q4_k_m.gguf', '2B FT v3.1'),
    ('base_4b', 'qwen3.5-4b-base-q4_k_m.gguf', '4B Base'),
    ('ft_4b', 'worldsim-v31-qwen3.5-4b-q4_k_m.gguf', '4B FT v3.1'),
    ('base_9b', 'qwen3.5-9b-base-q4_k_m.gguf', '9B Base'),
    ('ft_9b', 'worldsim-v31-qwen3.5-9b-q4_k_m.gguf', '9B FT v3.1'),
]

print('=== Model Inventory ===')
for key, filename, label in model_configs:
    path = GGUF_DIR / filename
    if path.exists():
        MODELS[key] = {'path': path, 'label': label, 'size_mb': round(path.stat().st_size / 1e6)}
        print(f"  ✅ {label:.<25} {MODELS[key]['size_mb']} MB")
    else:
        print(f"  ❌ {label:.<25} NOT FOUND")

print(f"\nConfigs: {len(situations)} situations, {len(personalities)} personalities, {len(emotions)} emotions")
print(f"Ready models: {len(MODELS)}")


## 3. Server Helpers & Test Prompts


In [ ]:
import urllib.request

SERVER_PORT = 8090
SERVER_URL = f'http://127.0.0.1:{SERVER_PORT}'


def start_server(model_path: Path, ctx_size: int = 2048, n_gpu_layers: int = 99):
    args = [
        str(LLAMA_SERVER), '-m', str(model_path),
        '--host', '127.0.0.1', '--port', str(SERVER_PORT),
        '-c', str(ctx_size), '-np', '1', '-ngl', str(n_gpu_layers),
        '-fa', 'on', '--jinja', '--no-webui',
        '--chat-template-kwargs', '{"enable_thinking": false}',
        '-ctk', 'q8_0', '-ctv', 'q8_0',
    ]
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for attempt in range(120):
        time.sleep(0.5)
        try:
            resp = urllib.request.urlopen(f'{SERVER_URL}/health', timeout=2)
            data = json.loads(resp.read())
            if data.get('status') == 'ok':
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            stderr = proc.stderr.read().decode(errors='ignore')[-300:]
            raise RuntimeError(f'Server died: {stderr}')
    proc.kill()
    raise RuntimeError('Server startup timeout')


def stop_server(proc) -> None:
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
            proc.wait()


def strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def query_model(messages, response_format=None, max_tokens: int = 256, temperature: float = 0):
    payload = {
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'stream': False,
    }
    if response_format:
        payload['response_format'] = response_format

    req = urllib.request.Request(
        f'{SERVER_URL}/v1/chat/completions',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read())
        content = data['choices'][0]['message']['content']
        return strip_think(content)
    except Exception as exc:
        return f'ERROR: {exc}'

rng = random.Random(42)

SYS_L3 = 'You are WorldSim logic assistant. Output JSON only. Follow [TEMP], [STRESS], [WORLD] context. Respect enum values and numeric ranges exactly.'
SYS_L4 = '너는 WorldSim 서사 도우미다. [TEMP], [STRESS], [WORLD]와 지정된 register를 반영해 JSON으로만 답하라.'
SYS_L5 = '너는 WorldSim 신탁 해석 도우미다. JSON으로만 답하라.'


def pick(items, n=1):
    return rng.sample(items, min(n, len(items)))


def temp_block(t):
    return f"NS={t['ns']} HA={t['ha']} RD={t['rd']} P={t['p']} type={t['id']}"


def personality_keywords(p):
    return ', '.join(p.get('keywords', []))


def json_schema(name: str, required: list[str], properties: dict) -> dict:
    return {
        'type': 'json_schema',
        'json_schema': {
            'name': name,
            'strict': True,
            'schema': {
                'type': 'object',
                'additionalProperties': False,
                'required': required,
                'properties': properties,
            },
        },
    }


emotion_ids = [e['id'] for e in emotions]
speaker_roles = ['elder', 'hunter', 'shaman', 'warrior', 'healer', 'gatherer', 'craftsman', 'chief', 'scout', 'observer']

b_schema = json_schema(
    'task_b',
    ['text_ko', 'text_en', 'register', 'emotion_expressed', 'intensity', 'mimetics', 'temperament_influence'],
    {
        'text_ko': {'type': 'string'},
        'text_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number'},
        'mimetics': {'type': 'array', 'items': {'type': 'string'}},
        'temperament_influence': {'type': 'string'},
    },
)
c_schema = json_schema(
    'task_c',
    ['speech_ko', 'speech_en', 'register', 'emotion_expressed', 'speaker_role', 'temperament_tone'],
    {
        'speech_ko': {'type': 'string'},
        'speech_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'speaker_role': {'type': 'string', 'enum': speaker_roles},
        'temperament_tone': {'type': 'string'},
    },
)
e_schema_for = lambda option_count: json_schema(
    'task_e',
    ['action_id', 'confidence', 'hint', 'personality_reasoning', 'temperament_factor'],
    {
        'action_id': {'type': 'integer', 'enum': list(range(option_count))},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'hint': {'type': 'string'},
        'personality_reasoning': {'type': 'string', 'enum': ['novelty_seeking', 'harm_avoidance', 'reward_dependence', 'persistence']},
        'temperament_factor': {'type': 'string'},
    },
)
f_schema = json_schema(
    'task_f',
    ['emotion', 'intensity', 'cause', 'previous_emotion', 'transition_type', 'temperament_amplifier'],
    {
        'emotion': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'cause': {'type': 'string'},
        'previous_emotion': {'type': 'string', 'enum': emotion_ids},
        'transition_type': {'type': 'string', 'enum': ['gradual', 'sudden', 'sustained']},
        'temperament_amplifier': {'type': 'string'},
    },
)
g_schema = json_schema(
    'task_g',
    ['interpretation_ko', 'interpretation_en', 'action_tendency', 'confidence', 'register', 'misinterpretation_type', 'temperament_bias'],
    {
        'interpretation_ko': {'type': 'string'},
        'interpretation_en': {'type': 'string'},
        'action_tendency': {'type': 'string', 'enum': ['mobilize', 'defend', 'wait', 'retreat', 'celebrate', 'mourn']},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'register': {'type': 'string', 'enum': ['haera', 'hao', 'hae']},
        'misinterpretation_type': {'type': 'string', 'enum': ['overconfident_literal', 'cautious_reversal', 'optimistic_expansion', 'passive_deferral', 'symbolic_abstraction']},
        'temperament_bias': {'type': 'string'},
    },
)
m_schema = json_schema(
    'task_m',
    ['decision_id', 'confidence', 'dissent_risk', 'reasoning', 'resource_commitment', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'reasoning': {'type': 'string'},
        'resource_commitment': {'type': 'string', 'enum': ['food', 'tools', 'labor', 'weapons', 'none']},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)
o_schema = json_schema(
    'task_o',
    ['public_claim', 'private_truth', 'deception_style', 'lie_degree', 'detection_risk', 'confidence'],
    {
        'public_claim': {'type': 'string'},
        'private_truth': {'type': 'string'},
        'deception_style': {'type': 'string', 'enum': ['evasion', 'half_truth', 'outright_lie', 'exaggeration', 'omission']},
        'lie_degree': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'detection_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
p_schema = json_schema(
    'task_p',
    ['retold_version', 'distortion_type', 'added_detail', 'dropped_detail', 'emotional_charge'],
    {
        'retold_version': {'type': 'string'},
        'distortion_type': {'type': 'string', 'enum': ['exaggeration', 'minimization', 'malicious_twist', 'emotional_coloring', 'detail_invention', 'source_confusion', 'faithful']},
        'added_detail': {'type': 'string'},
        'dropped_detail': {'type': 'string'},
        'emotional_charge': {'type': 'number', 'minimum': -1, 'maximum': 1},
    },
)
q_schema = json_schema(
    'task_q',
    ['trauma_response', 'behavioral_change', 'trigger_situation', 'intensity', 'duration', 'coping_mechanism'],
    {
        'trauma_response': {'type': 'string', 'enum': ['avoidance', 'overprotection', 'aggression', 'withdrawal', 'hypervigilance', 'ritual_coping', 'resilience']},
        'behavioral_change': {'type': 'string'},
        'trigger_situation': {'type': 'string'},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'duration': {'type': 'string', 'enum': ['short_term', 'long_term', 'permanent']},
        'coping_mechanism': {'type': 'string'},
    },
)
r_schema = json_schema(
    'task_r',
    ['action', 'counter_give', 'counter_want', 'reasoning', 'emotional_state', 'walk_away_threshold'],
    {
        'action': {'type': 'string', 'enum': ['accept', 'counter_offer', 'reject', 'walk_away', 'stall', 'bluff']},
        'counter_give': {'type': 'string'},
        'counter_want': {'type': 'string'},
        'reasoning': {'type': 'string'},
        'emotional_state': {'type': 'string', 'enum': emotion_ids},
        'walk_away_threshold': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
t_schema = json_schema(
    'task_t',
    ['decision_id', 'confidence', 'dissent_risk', 'minority_position', 'minority_action', 'spark_event', 'reasoning', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'minority_position': {'type': 'integer'},
        'minority_action': {'type': 'string', 'enum': ['comply', 'grumble', 'passive_resist', 'splinter', 'coup_attempt']},
        'spark_event': {'type': 'string', 'enum': ['food_shortage', 'battle_loss', 'oracle_conflict', 'leader_death', 'betrayal', 'resource_discovery']},
        'reasoning': {'type': 'string'},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)

ALL_PROMPTS = []
pair_counter = 0

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'B',
            'pair_id': f"B-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"B: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] B\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[기질 키워드] {temp['keywords']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[WORLD] default: 석기시대\n'
                '[출력 형식]\n'
                '{"text_ko": "해라체 2-3문장", "text_en": "English", "register": "haera", "emotion_expressed": "emotion", "intensity": 0.0, "mimetics": [], "temperament_influence": "str"}'
            ),
            'response_format': b_schema,
        })

for sit in pick(situations, 5):
    options = sit.get('action_options', sit.get('typical_actions', []))
    if not options:
        continue
    options_line = ' '.join(f"{i}:{o}" for i, o in enumerate(options))
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'E',
            'pair_id': f"E-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"E: {sit['id']} / {temp['id']}",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] E - Action Selection\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[EMOTION] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                f"[OPTIONS]\n{options_line}\n"
                '[OUTPUT FORMAT]\n'
                '{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'
            ),
            'response_format': e_schema_for(len(options)),
        })

for sit in pick(situations, 5):
    prev_emotion = rng.choice(emotions)
    for temp in [TEMPERAMENTS[2], TEMPERAMENTS[3]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'F',
            'pair_id': f"F-{len(ALL_PROMPTS) // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"F: {sit['id']} / {temp['id']} (prev={prev_emotion['id']})",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] F - Emotion Judgment\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[CURRENT EMOTION] {prev_emotion['id']}:{rng.choice(prev_emotion.get('intensities', [0.5]))}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                '[OUTPUT FORMAT]\n'
                f'{{"emotion": "one of 8", "intensity": 0.0-1.0, "cause": "sentence", "previous_emotion": "{prev_emotion["id"]}", "transition_type": "gradual|sudden|sustained", "temperament_amplifier": "str"}}'
            ),
            'response_format': f_schema,
        })

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    register = pers_register = 'haera'
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'C',
            'pair_id': None,
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"C: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] C\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.6]))}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[역할] warrior\n'
                '[출력 형식]\n'
                '{"speech_ko": "해라체 대사", "speech_en": "English", "register": "haera", "emotion_expressed": "emotion", "speaker_role": "role", "temperament_tone": "str"}'
            ),
            'response_format': c_schema,
        })

for sit in pick(situations, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'G',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'oracle',
        'scenario_id': sit['id'],
        'desc': f"G: {sit['id']} / {temp['id']}",
        'system': SYS_L5,
        'user_prompt': (
            f"[TASK] G - Oracle Interpretation\n[TEMP]\n{temp_block(temp)}\n"
            f"[기질 이름] {temp['id']}\n"
            f"[인물 성격] {temp['keywords']}\n"
            '[ORACLE]\n"큰물이 밀려오기 전에 높은 곳으로 가라"\n'
            f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
            '[역할] warrior\n'
            '[출력 형식]\n'
            '{"interpretation_ko": "해라체", "interpretation_en": "English", "action_tendency": "enum", "confidence": 0-1, "register": "haera", "misinterpretation_type": "enum", "temperament_bias": "str"}'
        ),
        'response_format': g_schema,
    })

for scenario in pick(deception_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    emotion = rng.choice(emotions)
    ALL_PROMPTS.append({
        'task': 'O',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"O: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] O - Deception\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.5]))}\n"
            f"[TRUE_STATE] {scenario['true_state']}\n"
            f"[PUBLIC_GOAL] {scenario['public_goal']}\n"
            f"[TARGET] {scenario['target']}\n"
            f"[DETECTION_CONTEXT] {scenario['detection_context']}\n"
            '[OUTPUT FORMAT]\n'
            '{"public_claim": "str", "private_truth": "str", "deception_style": "enum", "lie_degree": 0-1, "detection_risk": 0-1, "confidence": 0-1}'
        ),
        'response_format': o_schema,
    })

for scenario in pick(rumor_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'P',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"P: {scenario['id']} / {temp['id']} / {scenario['reteller_relationship']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] P - Rumor\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {rng.choice(emotions)['id']}:{round(rng.uniform(0.3, 0.8), 1)}\n"
            f"[ORIGINAL_FACT] {scenario['original_fact']}\n"
            f"[RETELLER_RELATIONSHIP] {scenario['reteller_relationship']}\n"
            '[OUTPUT FORMAT]\n'
            '{"retold_version": "str", "distortion_type": "enum", "added_detail": "str", "dropped_detail": "str", "emotional_charge": -1 to 1}'
        ),
        'response_format': p_schema,
    })

for scenario in pick(trauma_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'Q',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"Q: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] Q - Trauma\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] sadness:{round(rng.uniform(0.6, 0.95), 2)}\n"
            f"[TRAUMA_EVENT] {scenario['event']}\n"
            f"[TIME_SINCE] {scenario['time_since']}\n"
            f"[CURRENT_SITUATION] {scenario['current_situation']}\n"
            '[OUTPUT FORMAT]\n'
            '{"trauma_response": "enum", "behavioral_change": "str", "trigger_situation": "str", "intensity": 0-1, "duration": "enum", "coping_mechanism": "str"}'
        ),
        'response_format': q_schema,
    })

for scenario in pick(negotiation_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'R',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"R: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] R - Negotiate\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] anticipation:{round(rng.uniform(0.3, 0.7), 1)}\n"
            f"[CONTEXT] {scenario['context']}\n"
            f"[OUR_RESOURCES] {scenario['our_resources']}\n"
            f"[THEIR_RESOURCES] {scenario['their_resources']}\n"
            f"[OFFER_HISTORY] {scenario['offer_history']}\n"
            f"[POWER_BALANCE] {scenario['power_balance']}\n"
            '[OPTIONS]\n0:accept 1:counter_offer 2:reject 3:walk_away 4:stall 5:bluff\n'
            '[OUTPUT FORMAT]\n'
            '{"action": "enum", "counter_give": "str", "counter_want": "str", "reasoning": "str", "emotional_state": "emotion", "walk_away_threshold": 0-1}'
        ),
        'response_format': r_schema,
    })

for scenario in pick(group_situations, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o.get('ko', o.get('desc', ''))}" for o in opts) if opts else '0:agree 1:disagree 2:delay'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'M',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"M: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] M - Group Decision\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.3 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', scenario.get('desc', scenario['id']))}\n"
            f"[SITUATION] {scenario.get('situation', scenario.get('desc', scenario['id']))}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "reasoning": "str", "resource_commitment": "enum", "timeline": "enum"}'
        ),
        'response_format': m_schema,
    })

for scenario in pick(group_dissent_scenarios, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o['desc']}" for o in opts) if opts else '0:exile 1:trial 2:forgive'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'T',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"T: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] T - Group Dissent\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.4 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', '')}\n"
            f"[SITUATION] {scenario.get('situation', scenario['id'])}\n"
            f"[FACTION_HINT] {scenario.get('faction_hint', '')}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "minority_position": number, "minority_action": "enum", "spark_event": "enum", "reasoning": "str", "timeline": "enum"}'
        ),
        'response_format': t_schema,
    })

task_counts = Counter(prompt['task'] for prompt in ALL_PROMPTS)
print('=== Test Prompt Summary ===')
for task, count in sorted(task_counts.items()):
    paired = sum(1 for prompt in ALL_PROMPTS if prompt['task'] == task and prompt.get('pair_id'))
    print(f"  Task {task}: {count} prompts ({paired} paired)")
print(f"  Total: {len(ALL_PROMPTS)} prompts × {len(MODELS)} models = {len(ALL_PROMPTS) * len(MODELS)} inferences")


## 4. Run All Tests


In [ ]:
RESULTS = []
model_order = ['base_08b', 'ft_08b', 'base_2b', 'ft_2b', 'base_4b', 'ft_4b', 'base_9b', 'ft_9b']
available_models = [model for model in model_order if model in MODELS]

total_jobs = len(ALL_PROMPTS) * max(len(available_models), 1)
completed_jobs = 0
start_time = time.perf_counter()

for model_key in available_models:
    info = MODELS[model_key]
    ctx_size = 2048 if '9b' in model_key else 1024
    print()
    print('=' * 70)
    print(f"Testing model: {info['label']}")
    print('=' * 70)
    proc = start_server(info['path'], ctx_size=ctx_size)
    time.sleep(1)
    try:
        for prompt in ALL_PROMPTS:
            messages = [
                {'role': 'system', 'content': prompt['system']},
                {'role': 'user', 'content': prompt['user_prompt']},
            ]
            request_started = time.perf_counter()
            output = query_model(messages, response_format=prompt['response_format'], max_tokens=256, temperature=0)
            latency = (time.perf_counter() - request_started) * 1000
            json_valid = False
            parsed = None
            try:
                parsed = json.loads(output)
                json_valid = True
            except Exception:
                pass
            RESULTS.append({
                'model': model_key,
                'model_label': info['label'],
                'task': prompt['task'],
                'desc': prompt['desc'],
                'output': output,
                'parsed': parsed,
                'json_valid': json_valid,
                'latency_ms': round(latency),
                'pair_id': prompt.get('pair_id'),
                'temperament_id': prompt.get('temperament_id'),
                'personality_id': prompt.get('personality_id'),
                'scenario_id': prompt.get('scenario_id'),
            })
            completed_jobs += 1
            if completed_jobs % 10 == 0 or completed_jobs == total_jobs:
                elapsed = time.perf_counter() - start_time
                rate = completed_jobs / elapsed if elapsed > 0 else 0
                remaining = total_jobs - completed_jobs
                eta = remaining / rate if rate > 0 else 0
                print(
                    f"  Progress {completed_jobs}/{total_jobs} | "
                    f"task={prompt['task']} | model={model_key} | "
                    f"json={'Y' if json_valid else 'N'} | "
                    f"ETA={eta / 60:.1f} min",
                    flush=True,
                )
    finally:
        stop_server(proc)
        time.sleep(2)

print()
print(f'✅ Collected {len(RESULTS)} results')


## 5. Auto-Grade


In [ ]:
def auto_grade(result):
    grades = {}
    grades['json_valid'] = result['json_valid']

    if result['parsed']:
        values = [str(v) for v in result['parsed'].values() if isinstance(v, str)]
        placeholder_words = {'str', 'string', 'sentence', 'English', 'enum', 'number', '해라체', 'one of'}
        is_placeholder = any(value in placeholder_words for value in values) or any(len(value) < 4 for value in values if value)
        grades['not_placeholder'] = not is_placeholder
    else:
        grades['not_placeholder'] = False

    if result['parsed']:
        str_fields = [value for value in result['parsed'].values() if isinstance(value, str)]
        avg_len = sum(len(value) for value in str_fields) / max(len(str_fields), 1)
        grades['text_richness'] = avg_len > 15
    else:
        grades['text_richness'] = False

    if result['parsed']:
        grades['numeric_valid'] = True
        for key in ['confidence', 'intensity', 'lie_degree', 'detection_risk', 'dissent_risk', 'walk_away_threshold', 'emotional_charge']:
            if key in result['parsed']:
                value = result['parsed'][key]
                if isinstance(value, (int, float)):
                    if key == 'emotional_charge':
                        grades['numeric_valid'] = -1 <= value <= 1
                    else:
                        grades['numeric_valid'] = 0 <= value <= 1
                else:
                    grades['numeric_valid'] = False
                break
    else:
        grades['numeric_valid'] = False

    grades['score'] = sum([
        grades.get('json_valid', False),
        grades.get('not_placeholder', False),
        grades.get('text_richness', False),
        grades.get('numeric_valid', False),
    ])
    return grades


for row in RESULTS:
    row['grades'] = auto_grade(row)

print(f'✅ Graded {len(RESULTS)} results')

for row in RESULTS:
    row['grades'] = auto_grade(row)

print(f'✅ Graded {len(RESULTS)} results')


## 6. Per Model Summary


In [ ]:
print('=' * 90)
print('  EXTENDED COMPARISON — PER MODEL SUMMARY')
print('=' * 90)

model_order = ['base_08b', 'ft_08b', 'base_2b', 'ft_2b', 'base_4b', 'ft_4b', 'base_9b', 'ft_9b']
available = [model for model in model_order if model in MODELS]

print()
print(f"  {'Model':<20} {'JSON%':>6} {'!Placeholder%':>14} {'TextRich%':>10} {'Numeric%':>9} {'AvgScore':>9} {'AvgMs':>7}")
print(f"  {'-' * 77}")

for model_key in available:
    model_rows = [row for row in RESULTS if row['model'] == model_key]
    n = len(model_rows)
    json_pct = sum(1 for row in model_rows if row['grades']['json_valid']) / n * 100
    np_pct = sum(1 for row in model_rows if row['grades']['not_placeholder']) / n * 100
    tr_pct = sum(1 for row in model_rows if row['grades']['text_richness']) / n * 100
    num_pct = sum(1 for row in model_rows if row['grades']['numeric_valid']) / n * 100
    avg_score = sum(row['grades']['score'] for row in model_rows) / n
    avg_ms = sum(row['latency_ms'] for row in model_rows) / n
    label = MODELS[model_key]['label']
    print(f"  {label:<20} {json_pct:>5.0f}% {np_pct:>13.0f}% {tr_pct:>9.0f}% {num_pct:>8.0f}% {avg_score:>8.1f}/4 {avg_ms:>6.0f}ms")


## 7. Per Task Breakdown


In [ ]:
tasks = sorted(set(row['task'] for row in RESULTS))
model_order = ['base_08b', 'ft_08b', 'base_2b', 'ft_2b', 'base_4b', 'ft_4b', 'base_9b', 'ft_9b']
available = [model for model in model_order if model in MODELS]

print('=' * 90)
print('  PER TASK BREAKDOWN (Average Score /4)')
print('=' * 90)

print()
print(f"  {'Task':<6}", end='')
for model_key in available:
    print(f"  {MODELS[model_key]['label']:>14}", end='')
print()
print(f"  {'-' * 62}")

for task in tasks:
    print(f"  {task:<6}", end='')
    for model_key in available:
        rows = [row for row in RESULTS if row['task'] == task and row['model'] == model_key]
        if rows:
            avg = sum(row['grades']['score'] for row in rows) / len(rows)
            n = len(rows)
            symbol = '★' if avg >= 3.5 else '●' if avg >= 2.5 else '○' if avg >= 1.5 else '✗'
            print(f"  {avg:>5.1f} {symbol} (n={n})", end='')
        else:
            print(f"  {'—':>14}", end='')
    print()


## 8. Personality Consistency


In [ ]:
print('=' * 90)
print('  PERSONALITY CONSISTENCY — Paired Tests')
print('=' * 90)

pairs = defaultdict(list)
for row in RESULTS:
    pair_id = row.get('pair_id')
    if pair_id:
        pairs[(pair_id, row['model'])].append(row)

consistency_by_model = defaultdict(lambda: {'same': 0, 'different': 0, 'total': 0})

for (pair_id, model_key), pair_rows in sorted(pairs.items()):
    if len(pair_rows) != 2:
        continue
    first, second = pair_rows
    if not first['parsed'] or not second['parsed']:
        continue
    task = first['task']
    different = False
    if task == 'E':
        different = first['parsed'].get('action_id') != second['parsed'].get('action_id')
    elif task == 'B':
        different = (
            first['parsed'].get('emotion_expressed') != second['parsed'].get('emotion_expressed')
            or abs(first['parsed'].get('intensity', 0) - second['parsed'].get('intensity', 0)) > 0.1
        )
    elif task == 'F':
        different = first['parsed'].get('emotion') != second['parsed'].get('emotion')
    consistency_by_model[model_key]['total'] += 1
    if different:
        consistency_by_model[model_key]['different'] += 1
    else:
        consistency_by_model[model_key]['same'] += 1

model_order = ['base_08b', 'ft_08b', 'base_2b', 'ft_2b', 'base_4b', 'ft_4b', 'base_9b', 'ft_9b']
available = [model for model in model_order if model in MODELS]

print()
print(f"  {'Model':<20} {'Pairs':>6} {'Different':>10} {'Same':>6} {'Consistency%':>13}")
print(f"  {'-' * 58}")
for model_key in available:
    stats = consistency_by_model[model_key]
    total = stats['total']
    if total > 0:
        pct = stats['different'] / total * 100
        label = MODELS[model_key]['label']
        print(f"  {label:<20} {total:>6} {stats['different']:>10} {stats['same']:>6} {pct:>12.0f}%")

print()
print('  Higher % = model differentiates personalities better')
print('  Ideal: fine-tuned models should have higher consistency% than base')


## 9. Size Scaling Analysis


In [ ]:
print('=' * 90)
print('  SIZE SCALING ANALYSIS')
print('=' * 90)

ft_models = ['ft_08b', 'ft_2b', 'ft_4b', 'ft_9b']
sizes = {'ft_08b': 0.8, 'ft_2b': 2.0, 'ft_4b': 4.0, 'ft_9b': 9.0}
size_labels = {'ft_08b': '0.8B', 'ft_2b': '2B', 'ft_4b': '4B', 'ft_9b': '9B'}

eval_losses = {'ft_08b': 0.0838, 'ft_2b': 0.0748, 'ft_4b': 0.0673, 'ft_9b': 0.0646}

print()
print(f"  {'Size':<8} {'AvgScore':>9} {'JSON%':>6} {'!Placeholder%':>14} {'TextRich%':>10} {'AvgMs':>7} {'eval_loss':>10}")
print(f"  {'-' * 67}")
for model_key in ft_models:
    if model_key not in MODELS:
        continue
    model_rows = [row for row in RESULTS if row['model'] == model_key]
    if not model_rows:
        continue
    n = len(model_rows)
    avg_score = sum(row['grades']['score'] for row in model_rows) / n
    json_pct = sum(1 for row in model_rows if row['grades']['json_valid']) / n * 100
    np_pct = sum(1 for row in model_rows if row['grades']['not_placeholder']) / n * 100
    tr_pct = sum(1 for row in model_rows if row['grades']['text_richness']) / n * 100
    avg_ms = sum(row['latency_ms'] for row in model_rows) / n
    print(f"  {size_labels[model_key]:<8} {avg_score:>8.1f}/4 {json_pct:>5.0f}% {np_pct:>13.0f}% {tr_pct:>9.0f}% {avg_ms:>6.0f}ms {eval_losses.get(model_key, 0):>9.4f}")

print('\n  === KEY QUESTION: 4B vs 2B ===')
for model_key in ['ft_2b', 'ft_4b']:
    if model_key not in MODELS:
        continue
    model_rows = [row for row in RESULTS if row['model'] == model_key]
    if not model_rows:
        continue
    avg = sum(row['grades']['score'] for row in model_rows) / len(model_rows)
    avg_ms = sum(row['latency_ms'] for row in model_rows) / len(model_rows)
    print(f"  {size_labels[model_key]}: score={avg:.2f}/4, latency={avg_ms:.0f}ms, GGUF={MODELS[model_key]['size_mb']}MB")

print('\n  Tasks where 4B > 2B (score difference):')
for task in sorted(set(row['task'] for row in RESULTS)):
    ft2 = [row for row in RESULTS if row['task'] == task and row['model'] == 'ft_2b']
    ft4 = [row for row in RESULTS if row['task'] == task and row['model'] == 'ft_4b']
    if ft2 and ft4:
        s2 = sum(row['grades']['score'] for row in ft2) / len(ft2)
        s4 = sum(row['grades']['score'] for row in ft4) / len(ft4)
        delta = s4 - s2
        bar = '+' * int(abs(delta) * 10) if delta > 0 else '-' * int(abs(delta) * 10)
        marker = '⭐' if delta > 0.3 else '  '
        print(f"    {marker} Task {task}: 2B={s2:.1f} → 4B={s4:.1f} (Δ={delta:+.2f}) {bar}")


## 10. Save Results


In [ ]:
results_path = REPO_ROOT / 'outputs' / 'benchmarks' / 'v31_8model_comparison.json'
results_path.parent.mkdir(parents=True, exist_ok=True)

save_data = {
    'metadata': {
        'version': 'v3.1',
        'total_prompts': len(ALL_PROMPTS),
        'total_results': len(RESULTS),
        'models': {key: value['label'] for key, value in MODELS.items()},
    },
    'results': [{key: value for key, value in row.items() if key != 'parsed'} for row in RESULTS],
}

with results_path.open('w', encoding='utf-8') as handle:
    json.dump(save_data, handle, indent=2, ensure_ascii=False)

print(f'📁 Saved: {results_path} ({results_path.stat().st_size / 1024:.0f} KB)')
print('✅ v3.1 8-model comparison complete')
